# RQ1 - Model selection strategies

This notebook compares fixed holdout, three repeated holdouts, and three-fold cross-validation using the original continuous representation. The independent test scores are reported, but they are never used to select a configuration.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if ROOT.name == 'experiments':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.metrics import f1_score
from sklearn.model_selection import ParameterGrid, StratifiedKFold, train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from src.models import hyperparameters, models
from src.preprocessing import categoric_cols, numeric_cols, split_data

In [ ]:
clean_path = ROOT / 'data' / 'preprocessed' / 'adult_cleaned.csv'
clean = pd.read_csv(clean_path)
development, test = split_data(clean)

X = development.drop(columns='income')
y = development['income']
X_test = test.drop(columns='income')
y_test = test['income']

categorical_features = [col for col in categoric_cols if col != 'income']

print('Cleaned data:', clean.shape)
print('Development data:', development.shape)
print('Independent test data:', test.shape)

In [ ]:
def holdout(seed):
    return train_test_split(
        np.arange(len(X)),
        test_size=0.20,
        random_state=seed,
        stratify=y
    )


cross_validation = StratifiedKFold(
    n_splits=3,
    shuffle=True,
    random_state=42
)

strategies = {
    'holdout': [holdout(42)],
    'repeated': [holdout(seed) for seed in [42, 43, 44]],
    'cv': list(cross_validation.split(X, y))
}

for name, partitions in strategies.items():
    print(name, 'partitions:', len(partitions))

In [ ]:
def prepare_features(train_idx, val_idx, classifier_name):
    train_features = X.iloc[train_idx]
    val_features = X.iloc[val_idx]

    transformer = ColumnTransformer([
        (
            'numeric',
            StandardScaler() if classifier_name == 'kNN' else 'passthrough',
            numeric_cols
        ),
        (
            'categorical',
            OneHotEncoder(handle_unknown='ignore', sparse_output=False),
            categorical_features
        )
    ])

    train_matrix = transformer.fit_transform(train_features)
    val_matrix = transformer.transform(val_features)
    test_matrix = transformer.transform(X_test)

    return train_matrix, val_matrix, test_matrix

In [ ]:
def model_f1(model, features, target):
    predictions = model.predict(features)

    return f1_score(
        target,
        predictions,
        pos_label='>50K',
        zero_division=0
    )


def run_classifier(name):
    configurations = list(ParameterGrid(hyperparameters[name]))
    parameter_names = list(hyperparameters[name])
    rows = []

    for config_id, parameters in enumerate(configurations, start=1):
        row = {'ID': config_id}

        for parameter_name in parameter_names:
            value = parameters[parameter_name]
            row[parameter_name] = 'None' if value is None else value

        rows.append(row)

    table = pd.DataFrame(rows)

    for strategy_name, partitions in strategies.items():
        train_scores = [[] for _ in configurations]
        val_scores = [[] for _ in configurations]
        test_scores = [[] for _ in configurations]

        for train_idx, val_idx in partitions:
            train_matrix, val_matrix, test_matrix = prepare_features(
                train_idx,
                val_idx,
                name
            )

            train_target = y.iloc[train_idx]
            val_target = y.iloc[val_idx]

            for config_index, parameters in enumerate(configurations):
                model = clone(models[name])
                model.set_params(**parameters)

                if 'random_state' in model.get_params():
                    model.set_params(random_state=42)

                model.fit(train_matrix, train_target)

                train_scores[config_index].append(
                    model_f1(model, train_matrix, train_target)
                )
                val_scores[config_index].append(
                    model_f1(model, val_matrix, val_target)
                )
                test_scores[config_index].append(
                    model_f1(model, test_matrix, y_test)
                )

        table[f'{strategy_name}_train_F1'] = [
            np.mean(scores) for scores in train_scores
        ]
        table[f'{strategy_name}_val_F1'] = [
            np.mean(scores) for scores in val_scores
        ]
        table[f'{strategy_name}_test_F1'] = [
            np.mean(scores) for scores in test_scores
        ]

        print(name, '-', strategy_name, 'complete')

    return table

## Decision Tree

In [ ]:
dt_results = run_classifier('Decision_Tree')
display(dt_results)

## k-NN

In [ ]:
knn_results = run_classifier('kNN')
display(knn_results)

## Random Forest

In [ ]:
rf_results = run_classifier('Random_Forest')
display(rf_results)

## Selected configurations and formatted tables

In [ ]:
results = {
    'Decision_Tree': dt_results,
    'kNN': knn_results,
    'Random_Forest': rf_results
}

selection_scores = {
    'holdout validation': 'holdout_val_F1',
    'repeated holdout validation': 'repeated_val_F1',
    'cross-validation': 'cv_val_F1',
    'optimistic holdout training': 'holdout_train_F1'
}

for name, table in results.items():
    print('\n', name)

    for description, column in selection_scores.items():
        best_score = table[column].max()
        best_ids = table.loc[table[column] == best_score, 'ID'].tolist()
        print(description, '- IDs:', best_ids, 'F1:', round(best_score, 4))

    f1_columns = [col for col in table.columns if col.endswith('_F1')]
    formatted_table = (
        table.style
        .format(precision=4, subset=f1_columns)
        .highlight_max(subset=f1_columns, props='font-weight: bold;')
        .set_caption(f'{name}: model-selection F1-scores')
    )
    display(formatted_table)

## Save result tables

In [ ]:
output_directory = ROOT / 'results' / 'tables' / 'RQ1'
output_directory.mkdir(parents=True, exist_ok=True)

for name, table in results.items():
    output_path = output_directory / f'{name}_results.csv'
    table.to_csv(output_path, index=False)
    print('Saved:', output_path)